In [2]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026\Graduate_Dataset\all_sources"
)

FOLDERS = {
    "gender": ROOT / "014_GDS",
    "wdi": ROOT / "002_WDI",
    "join": ROOT / "086_JON",
}

for name, folder in FOLDERS.items():
    print(f"{name}: exists={folder.exists()}")

gender: exists=True
wdi: exists=True
join: exists=True


In [3]:
catalogs = {
    name: pd.read_csv(
        folder / "_indicators.csv",
        dtype="string",
        keep_default_na=False,
    )
    for name, folder in FOLDERS.items()
}

for name, catalog in catalogs.items():
    print(f"{name}: {len(catalog):,} indicators")

display(catalogs["gender"].head())

gender: 1,411 indicators
wdi: 1,498 indicators
join: 746 indicators


,id,name,unit,source_org,topics,definition
0,account.t.d,Account (% age 15+),,Global Findex Database,,The percentage of respondents who report havin...
1,account.t.d.1,"Account, female (% age 15+)",,Global Findex Database,,The percentage of respondents who report havin...
2,account.t.d.2,"Account, male (% age 15+)",,Global Findex Database,,The percentage of respondents who report havin...
3,borrow.any.t.d,Borrowed any money (% age 15+),,Global Findex Database,,The percentage of respondents who report borro...
4,borrow.any.t.d.1,"Borrowed any money, women (% age 15+)",,Global Findex Database,,The percentage of respondents who report borro...


In [4]:
SELECTED = {
    "gender": [
        "SG.TIM.UWRK.FE",       # Women's unpaid domestic and care work time
        "SG.TIM.UWRK.MA",       # Men's unpaid domestic and care work time
    ],
    "wdi": [
        "EG.ELC.ACCS.ZS",       # Electricity access
    ],
    "join": [
        "JI.EMP.TOTL.SP.FE.ZS", # Women's employment rate, age 15–64
    ],
}

# Show the selected indicators
for source, codes in SELECTED.items():
    print(f"\n--- {source.upper()} ---")

    display(
        catalogs[source].loc[
            catalogs[source]["id"].isin(codes),
            ["id", "name", "unit", "definition"],
        ]
    )


--- GENDER ---


,id,name,unit,definition
886,SG.TIM.UWRK.FE,Proportion of time spent on unpaid domestic an...,,The average time women spend on household prov...
887,SG.TIM.UWRK.MA,Proportion of time spent on unpaid domestic an...,,The average time men spend on household provis...



--- WDI ---


,id,name,unit,definition
210,EG.ELC.ACCS.ZS,Access to electricity (% of population),,Access to electricity is the percentage of pop...



--- JOIN ---


,id,name,unit,definition
381,JI.EMP.TOTL.SP.FE.ZS,"Employment to population ratio, aged 15-64, fe...",,


In [5]:
def read_indicator(source, code):
    path = FOLDERS[source] / f"{code}.csv.gz"

    if not path.is_file():
        raise FileNotFoundError(path)

    # Read the CSV file with all columns as strings
    return pd.read_csv(
        path,
        compression="gzip",
        dtype="string",
        keep_default_na=False,
    )


data = {}

for source, codes in SELECTED.items():
    for code in codes:
        data[(source, code)] = read_indicator(source, code)

        print(f"\n--- {source} / {code} ---")
        display(data[(source, code)].head())


--- gender / SG.TIM.UWRK.FE ---


,country_id,country,iso3,date,value,obs_status
0,AL,Albania,ALB,2011,21.73611,
1,DZ,Algeria,DZA,2012,21.66667,
2,AR,Argentina,ARG,2010,17.5,
3,AR,Argentina,ARG,2005,16.73611,
4,AM,Armenia,ARM,2008,21.65833,



--- gender / SG.TIM.UWRK.MA ---


,country_id,country,iso3,date,value,obs_status
0,AL,Albania,ALB,2011,3.47222,
1,DZ,Algeria,DZA,2012,3.75,
2,AR,Argentina,ARG,2010,6.59722,
3,AR,Argentina,ARG,2005,6.11111,
4,AM,Armenia,ARM,2008,4.37083,



--- wdi / EG.ELC.ACCS.ZS ---


,country_id,country,iso3,date,value,obs_status
0,ZH,Africa Eastern and Southern,AFE,2024,52.5981705163589,
1,ZH,Africa Eastern and Southern,AFE,2023,50.6675157944854,
2,ZH,Africa Eastern and Southern,AFE,2022,48.7420429375177,
3,ZH,Africa Eastern and Southern,AFE,2021,48.1272110400485,
4,ZH,Africa Eastern and Southern,AFE,2020,46.2823709455637,



--- join / JI.EMP.TOTL.SP.FE.ZS ---


,country_id,country,iso3,date,value,obs_status
0,ALB,Albania,,2019,0.408310920000076,
1,ALB,Albania,,2018,0.404861092567444,
2,ALB,Albania,,2017,0.438733637332916,
3,ALB,Albania,,2016,0.392252743244171,
4,ALB,Albania,,2015,0.362558454275131,


In [6]:
def inspect_indicator(source, code, df):
    year_text = df["date"].str.strip()
    value_text = df["value"].str.strip()

    years = pd.to_numeric(
        year_text.replace("", pd.NA), errors="coerce"
    )
    values = pd.to_numeric(
        value_text.replace("", pd.NA), errors="coerce"
    )

    valid_country_ids = df.loc[
        df["country_id"].str.strip().ne(""), "country_id"
    ]

    return {
        "source": source,
        "indicator": code,
        "rows": len(df),
        # Number of unique entities (countries, regions, income groups)
        "entity_ids": valid_country_ids.nunique(),
        "first_year": years.min(),
        "last_year": years.max(),
        "blank_country_id": df["country_id"].str.strip().eq("").sum(),
        "blank_iso3": df["iso3"].str.strip().eq("").sum(),
        "blank_value": value_text.eq("").sum(),
        "unparsed_year": (year_text.ne("") & years.isna()).sum(),
        "unparsed_value": (value_text.ne("") & values.isna()).sum(),
        "value_min": values.min(),
        "value_max": values.max(),
        "duplicate_country_year": df.duplicated(
            subset=["country_id", "date"]
        ).sum(),
    }


checks = pd.DataFrame([
    inspect_indicator(source, code, df)
    for (source, code), df in data.items()
])

display(checks)

,source,indicator,rows,entity_ids,first_year,last_year,blank_country_id,blank_iso3,blank_value,unparsed_year,unparsed_value,value_min,value_max,duplicate_country_year
0,gender,SG.TIM.UWRK.FE,66,27,2000,2024,0,0,0,0,0,10.833330,25.416670,0
1,gender,SG.TIM.UWRK.MA,66,27,2000,2024,0,0,0,0,0,1.250000,12.638890,0
2,wdi,EG.ELC.ACCS.ZS,8097,263,1990,2024,0,117,0,0,0,0.533899,100.000000,0
3,join,JI.EMP.TOTL.SP.FE.ZS,1639,164,1970,2021,0,1639,0,0,0,0.078908,0.980855,0


In [7]:
female_time = data[("gender", "SG.TIM.UWRK.FE")]

year_counts = (
    female_time.groupby("date")
    .size()
    .rename("observations")
    .sort_index()
)

display(year_counts.to_frame())

,observations
date,
2000,2
2001,3
2002,2
2004,2
2005,3
2006,1
2007,2
2008,4
2009,4


In [8]:
entity_counts = (
    female_time.groupby(["country_id", "country"])
    .size()
    .rename("observations")
    .reset_index()
    .sort_values(
        ["observations", "country"],
        ascending=[False, True],
    )
)

display(entity_counts)

,country_id,country,observations
21,EC,Ecuador,11
9,BR,Brazil,7
12,CA,Canada,4
4,AU,Australia,3
6,BE,Belgium,3
7,BG,Bulgaria,3
15,CO,Colombia,3
2,AR,Argentina,2
1,AM,Armenia,2
3,AT,Austria,2


In [9]:
import json
import time
from urllib.parse import urlencode
from urllib.request import urlopen

def fetch_time_indicator(code):
    """Read all pages of a time series indicator from the World Bank API."""
    records = []
    page = 1
    expected_total = None

    while True:
        params = urlencode({
            "source": 14,
            "format": "json",
            "per_page": 1000,
            "page": page,
        })
        url = (
            f"https://api.worldbank.org/v2/country/all/"
            f"indicator/{code}?{params}"
        )

        # Retry up to three times in case of network or API failures
        for attempt in range(3):
            try:
                with urlopen(url, timeout=120) as response:
                    payload = json.load(response)

                if (
                    not isinstance(payload, list)
                    or len(payload) != 2
                    or not isinstance(payload[0], dict)
                    or "pages" not in payload[0]
                ):
                    raise ValueError(f"API return value is invalid：{str(payload)[:400]}")

                break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 * (attempt + 1))

        metadata, rows = payload
        total = int(metadata["total"])
        pages = int(metadata["pages"])

        if expected_total is None:
            expected_total = total
        elif total != expected_total:
            raise RuntimeError("During the download, the total number of API records has changed. Please rerun the script.")

        records.extend(rows or [])
        print(
            f"{code}: page {page}/{pages}, "
            f"Items: {len(records):,}/{total:,} "
        )

        if page >= pages:
            break
        page += 1

    if len(records) != expected_total:
        raise RuntimeError(
            f"Expected {expected_total} records, but got {len(records)}"
        )

    # Filter out records with null values
    nonnull = pd.DataFrame([
        {
            "country_id": str(row["country"]["id"]),
            "country": row["country"]["value"],
            "iso3": row.get("countryiso3code", ""),
            "date": str(row["date"]),
            "value": row["value"],
        }
        for row in records
        if row.get("value") is not None
    ], columns=["country_id", "country", "iso3", "date", "value"])

    return nonnull, expected_total


audit_results = {}
summary_rows = []

for code in ["SG.TIM.UWRK.FE", "SG.TIM.UWRK.MA"]:
    local = data[("gender", code)].copy()
    remote, total = fetch_time_indicator(code)

    keys = ["country_id", "date"]

    # Compare the remote data with the local data based on country and date
    comparison = (
        remote[keys].drop_duplicates()
        .merge(
            local[keys].drop_duplicates(),
            on=keys,
            how="outer",
            indicator=True,
            validate="one_to_one",
        )
    )

    missing_keys = comparison.loc[
        comparison["_merge"].eq("left_only"), keys
    ]
    local_only_keys = comparison.loc[
        comparison["_merge"].eq("right_only"), keys
    ]

    missing_local = remote.merge(missing_keys, on=keys, how="inner")
    local_only = local.merge(local_only_keys, on=keys, how="inner")

    audit_results[code] = {
        "api_data": remote,
        "missing_local": missing_local,
        "local_only": local_only,
    }

    summary_rows.append({
        "indicator": code,
        "api_total_including_null": total,
        "api_nonnull_rows": len(remote),
        "api_entities": remote["country_id"].nunique(),
        "local_rows": len(local),
        "missing_local_keys": len(missing_keys),
        "local_only_keys": len(local_only_keys),
        "api_duplicate_keys": remote.duplicated(keys).sum(),
    })

    print(f"\n{code}: API has, but local doesn't (first 10)")
    display(missing_local.head(10))

audit_summary = pd.DataFrame(summary_rows)
display(audit_summary)

SG.TIM.UWRK.FE: page 1/18, Items: 1,000/17,424 
SG.TIM.UWRK.FE: page 2/18, Items: 2,000/17,424 
SG.TIM.UWRK.FE: page 3/18, Items: 3,000/17,424 
SG.TIM.UWRK.FE: page 4/18, Items: 4,000/17,424 
SG.TIM.UWRK.FE: page 5/18, Items: 5,000/17,424 
SG.TIM.UWRK.FE: page 6/18, Items: 6,000/17,424 
SG.TIM.UWRK.FE: page 7/18, Items: 7,000/17,424 
SG.TIM.UWRK.FE: page 8/18, Items: 8,000/17,424 
SG.TIM.UWRK.FE: page 9/18, Items: 9,000/17,424 
SG.TIM.UWRK.FE: page 10/18, Items: 10,000/17,424 
SG.TIM.UWRK.FE: page 11/18, Items: 11,000/17,424 
SG.TIM.UWRK.FE: page 12/18, Items: 12,000/17,424 
SG.TIM.UWRK.FE: page 13/18, Items: 13,000/17,424 
SG.TIM.UWRK.FE: page 14/18, Items: 14,000/17,424 
SG.TIM.UWRK.FE: page 15/18, Items: 15,000/17,424 
SG.TIM.UWRK.FE: page 16/18, Items: 16,000/17,424 
SG.TIM.UWRK.FE: page 17/18, Items: 17,000/17,424 
SG.TIM.UWRK.FE: page 18/18, Items: 17,424/17,424 

SG.TIM.UWRK.FE: API has, but local doesn't (first 10)


,country_id,country,iso3,date,value


SG.TIM.UWRK.MA: page 1/18, Items: 1,000/17,424 
SG.TIM.UWRK.MA: page 2/18, Items: 2,000/17,424 
SG.TIM.UWRK.MA: page 3/18, Items: 3,000/17,424 
SG.TIM.UWRK.MA: page 4/18, Items: 4,000/17,424 
SG.TIM.UWRK.MA: page 5/18, Items: 5,000/17,424 
SG.TIM.UWRK.MA: page 6/18, Items: 6,000/17,424 
SG.TIM.UWRK.MA: page 7/18, Items: 7,000/17,424 
SG.TIM.UWRK.MA: page 8/18, Items: 8,000/17,424 
SG.TIM.UWRK.MA: page 9/18, Items: 9,000/17,424 
SG.TIM.UWRK.MA: page 10/18, Items: 10,000/17,424 
SG.TIM.UWRK.MA: page 11/18, Items: 11,000/17,424 
SG.TIM.UWRK.MA: page 12/18, Items: 12,000/17,424 
SG.TIM.UWRK.MA: page 13/18, Items: 13,000/17,424 
SG.TIM.UWRK.MA: page 14/18, Items: 14,000/17,424 
SG.TIM.UWRK.MA: page 15/18, Items: 15,000/17,424 
SG.TIM.UWRK.MA: page 16/18, Items: 16,000/17,424 
SG.TIM.UWRK.MA: page 17/18, Items: 17,000/17,424 
SG.TIM.UWRK.MA: page 18/18, Items: 17,424/17,424 

SG.TIM.UWRK.MA: API has, but local doesn't (first 10)


,country_id,country,iso3,date,value


,indicator,api_total_including_null,api_nonnull_rows,api_entities,local_rows,missing_local_keys,local_only_keys,api_duplicate_keys
0,SG.TIM.UWRK.FE,17424,66,27,66,0,0,0
1,SG.TIM.UWRK.MA,17424,66,27,66,0,0,0


In [10]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\all_sources"
)

FOLDERS = {
    "gender": ROOT / "014_GDS",
    "wdi": ROOT / "002_WDI",
    "join": ROOT / "086_JON",
}

catalogs = {
    source: pd.read_csv(
        folder / "_indicators.csv",
        dtype="string",
        keep_default_na=False,
    )
    for source, folder in FOLDERS.items()
}

print("Indicator catalogs loaded.")

Indicator catalogs loaded.


In [11]:
candidate_specs = [
    # Time burden
    ("Time burden", "gender", "SG.TIM.UWRK.FE",
     "Female unpaid domestic and care work"),
    ("Time burden", "gender", "SG.TIM.UWRK.MA",
     "Male unpaid domestic and care work"),

    # Economic opportunity
    ("Economic opportunity", "join", "JI.EMP.TOTL.SP.FE.ZS",
     "Female employment-to-population ratio"),
    ("Economic opportunity", "join", "JI.EMP.TOTL.SP.MA.ZS",
     "Male employment-to-population ratio"),
    ("Economic opportunity", "gender", "SL.TLF.CACT.FE.ZS",
     "Female labor force participation"),
    ("Economic opportunity", "gender", "SL.TLF.CACT.MA.ZS",
     "Male labor force participation"),

    # Infrastructure
    ("Infrastructure", "wdi", "EG.ELC.ACCS.ZS",
     "Access to electricity"),
    ("Infrastructure", "wdi", "SH.H2O.BASW.ZS",
     "Access to basic drinking water services"),
    ("Infrastructure", "wdi", "EG.CFT.ACCS.ZS",
     "Access to clean cooking fuels and technologies"),

    # Country context
    ("Country context", "wdi", "NY.GDP.PCAP.PP.KD",
     "GDP per capita, PPP, constant international dollars"),
    ("Country context", "wdi", "SP.DYN.TFRT.IN",
     "Fertility rate"),
    ("Country context", "wdi", "SP.URB.TOTL.IN.ZS",
     "Urban population share"),
]

rows = []

for category, source, code, purpose in candidate_specs:
    matches = catalogs[source].loc[
        catalogs[source]["id"].eq(code)
    ]

    row = {
        "category": category,
        "source": source,
        "indicator": code,
        "purpose": purpose,
        "catalog_matches": len(matches),
        "file_exists": (
            FOLDERS[source] / f"{code}.csv.gz"
        ).is_file(),
    }

    # Only use metadata when the catalog match is unambiguous.
    for field in ["name", "unit", "source_org", "definition"]:
        row[field] = (
            matches.iloc[0].get(field, "")
            if len(matches) == 1
            else ""
        )

    rows.append(row)

indicator_dictionary = pd.DataFrame(rows)

display(
    indicator_dictionary[
        [
            "category",
            "source",
            "indicator",
            "catalog_matches",
            "file_exists",
            "name",
        ]
    ]
)

,category,source,indicator,catalog_matches,file_exists,name
0,Time burden,gender,SG.TIM.UWRK.FE,1,True,Proportion of time spent on unpaid domestic an...
1,Time burden,gender,SG.TIM.UWRK.MA,1,True,Proportion of time spent on unpaid domestic an...
2,Economic opportunity,join,JI.EMP.TOTL.SP.FE.ZS,1,True,"Employment to population ratio, aged 15-64, fe..."
3,Economic opportunity,join,JI.EMP.TOTL.SP.MA.ZS,1,True,"Employment to population ratio, aged 15-64, ma..."
4,Economic opportunity,gender,SL.TLF.CACT.FE.ZS,1,True,"Labor force participation rate, female (% of f..."
5,Economic opportunity,gender,SL.TLF.CACT.MA.ZS,1,True,"Labor force participation rate, male (% of mal..."
6,Infrastructure,wdi,EG.ELC.ACCS.ZS,1,True,Access to electricity (% of population)
7,Infrastructure,wdi,SH.H2O.BASW.ZS,1,True,People using at least basic drinking water ser...
8,Infrastructure,wdi,EG.CFT.ACCS.ZS,1,True,Access to clean fuels and technologies for coo...
9,Country context,wdi,NY.GDP.PCAP.PP.KD,1,True,"GDP per capita, PPP (constant 2021 internation..."


In [12]:
for _, row in indicator_dictionary.iterrows():
    print("=" * 80)
    print(f"Category: {row['category']}")
    print(f"Database: {row['source']}")
    print(f"Indicator: {row['indicator']}")
    print(f"Official name: {row['name'] or '[Missing]'}")
    print(f"Unit field: {row['unit'] or '[Not provided]'}")
    print(f"Original source: {row['source_org'] or '[Not provided]'}")
    print("\nDefinition:")
    print(row["definition"] or "[Not provided — requires verification]")
    print()

Category: Time burden
Database: gender
Indicator: SG.TIM.UWRK.FE
Official name: Proportion of time spent on unpaid domestic and care work, female (% of 24 hour day)
Unit field: [Not provided]
Original source: National statistical offices or national database and publications, United Nations (UN), uri: https://unstats.un.org/sdgs/dataportal/database, note: Indicator code from the original source: SH_FPL_INFM; 	Indicator name from the original source: Proportion of time spent on unpaid domestic chores and care work, by sex, age and location (%), publisher: UN Statistics Division, type: Excel, date accessed: 46190, date published: 2026-03-27

Definition:
The average time women spend on household provision of services for own consumption. Data are expressed as a proportion of time in a day. Domestic and care work includes food preparation, dishwashing, cleaning and upkeep of a dwelling, laundry, ironing, gardening, caring for pets, shopping, installation, servicing and repair of personal a

In [13]:
from pathlib import Path
import pandas as pd

# Requires FOLDERS and indicator_dictionary from the previous cells.
def read_raw_indicator(source, code):
    path = FOLDERS[source] / f"{code}.csv.gz"

    return pd.read_csv(
        path,
        compression="gzip",
        dtype="string",
        keep_default_na=False,
    )


female_raw = read_raw_indicator("gender", "SG.TIM.UWRK.FE")

country_map = (
    female_raw[["country_id", "iso3", "country"]]
    .apply(lambda column: column.str.strip())
    .drop_duplicates()
    .reset_index(drop=True)
)

if country_map[["country_id", "iso3"]].eq("").any().any():
    raise ValueError("The anchor country mapping contains empty codes.")

if (
    country_map["country_id"].duplicated().any()
    or country_map["iso3"].duplicated().any()
):
    raise ValueError("The anchor country mapping is not one-to-one.")

# Accept either code format for entities in the anchor sample.
code_to_iso3 = {}

for row in country_map.itertuples(index=False):
    for code in [row.country_id, row.iso3]:
        if code in code_to_iso3 and code_to_iso3[code] != row.iso3:
            raise ValueError(f"Conflicting country code: {code}")

        code_to_iso3[code] = row.iso3

anchor_iso3 = set(country_map["iso3"])

anchor = female_raw[["iso3", "country", "date"]].copy()
anchor["iso3"] = anchor["iso3"].str.strip()
anchor["year"] = pd.to_numeric(
    anchor.pop("date"), errors="raise"
).astype("Int64")

KEYS = ["iso3", "year"]

if anchor.duplicated(KEYS).any():
    raise ValueError("Duplicate country-year keys in the anchor sample.")

print(f"Anchor observations: {len(anchor)}")
print(f"Anchor entities: {anchor['iso3'].nunique()}")
display(country_map)

Anchor observations: 66
Anchor entities: 27


,country_id,iso3,country
0,AL,ALB,Albania
1,DZ,DZA,Algeria
2,AR,ARG,Argentina
3,AM,ARM,Armenia
4,AU,AUS,Australia
5,AT,AUT,Austria
6,AZ,AZE,Azerbaijan
7,BY,BLR,Belarus
8,BE,BEL,Belgium
9,BT,BTN,Bhutan


In [14]:
def prepare_for_matching(source, code):
    raw = read_raw_indicator(source, code)

    country_id = raw["country_id"].str.strip()
    reported_iso3 = raw["iso3"].str.strip()

    mapped_from_id = country_id.map(code_to_iso3)
    reported_in_anchor = reported_iso3.isin(anchor_iso3)

    conflict = (
        reported_iso3.ne("")
        & mapped_from_id.notna()
        & reported_iso3.ne(mapped_from_id).fillna(False)
    )

    if conflict.any():
        display(raw.loc[conflict].head(10))
        raise ValueError(f"Conflicting country codes in {code}")

    resolved_iso3 = reported_iso3.where(
        reported_in_anchor, mapped_from_id
    )

    # Restrict this check to entities present in the anchor sample.
    keep = resolved_iso3.isin(anchor_iso3)

    prepared = raw.loc[keep, ["date", "value"]].copy()
    prepared["iso3"] = resolved_iso3.loc[keep]

    prepared["year"] = pd.to_numeric(
        prepared.pop("date").str.strip().replace("", pd.NA),
        errors="raise",
    ).astype("Int64")

    prepared["value"] = pd.to_numeric(
        prepared["value"].str.strip().replace("", pd.NA),
        errors="raise",
    )

    if prepared["year"].isna().any():
        raise ValueError(f"Missing years in {code}")

    if prepared.duplicated(KEYS).any():
        duplicates = prepared.loc[
            prepared.duplicated(KEYS, keep=False)
        ]
        display(duplicates.head(10))
        raise ValueError(f"Duplicate country-year keys in {code}")

    return prepared[["iso3", "year", "value"]]


prepared_indicators = {}

for row in indicator_dictionary.itertuples(index=False):
    if row.catalog_matches != 1 or not row.file_exists:
        raise ValueError(f"Unavailable or ambiguous indicator: {row.indicator}")

    key = (row.source, row.indicator)
    prepared_indicators[key] = prepare_for_matching(*key)

print(f"Prepared indicators: {len(prepared_indicators)}")

Prepared indicators: 12


In [15]:
coverage_rows = []
missing_details = {}

# Keep every anchor observation, even if other indicators are missing.
analysis_preview = anchor.copy()

for row in indicator_dictionary.itertuples(index=False):
    key = (row.source, row.indicator)
    prepared = prepared_indicators[key]

    matched = anchor.merge(
        prepared,
        on=KEYS,
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    valid = matched["value"].notna()
    absent_key = matched["_merge"].eq("left_only")
    empty_value = matched["_merge"].eq("both") & ~valid

    coverage_rows.append({
        "category": row.category,
        "source": row.source,
        "indicator": row.indicator,
        "anchor_observations": len(anchor),
        "matched_observations": int(valid.sum()),
        "missing_observations": int((~valid).sum()),
        "absent_country_year": int(absent_key.sum()),
        "empty_value": int(empty_value.sum()),
        "match_rate_pct": round(valid.mean() * 100, 1),
    })

    missing_details[key] = matched.loc[
        ~valid, ["iso3", "country", "year"]
    ].copy()

    column_name = f"{row.source}__{row.indicator}"

    analysis_preview = analysis_preview.merge(
        prepared.rename(columns={"value": column_name}),
        on=KEYS,
        how="left",
        validate="one_to_one",
    )

coverage = pd.DataFrame(coverage_rows)

with pd.option_context("display.max_columns", None, "display.width", 180):
    display(coverage)

print(f"Preview rows retained: {len(analysis_preview)}")
display(analysis_preview.head())

,category,source,indicator,anchor_observations,matched_observations,missing_observations,absent_country_year,empty_value,match_rate_pct
0,Time burden,gender,SG.TIM.UWRK.FE,66,66,0,0,0,100.0
1,Time burden,gender,SG.TIM.UWRK.MA,66,66,0,0,0,100.0
2,Economic opportunity,join,JI.EMP.TOTL.SP.FE.ZS,66,33,33,33,0,50.0
3,Economic opportunity,join,JI.EMP.TOTL.SP.MA.ZS,66,33,33,33,0,50.0
4,Economic opportunity,gender,SL.TLF.CACT.FE.ZS,66,66,0,0,0,100.0
5,Economic opportunity,gender,SL.TLF.CACT.MA.ZS,66,66,0,0,0,100.0
6,Infrastructure,wdi,EG.ELC.ACCS.ZS,66,66,0,0,0,100.0
7,Infrastructure,wdi,SH.H2O.BASW.ZS,66,66,0,0,0,100.0
8,Infrastructure,wdi,EG.CFT.ACCS.ZS,66,60,6,6,0,90.9
9,Country context,wdi,NY.GDP.PCAP.PP.KD,66,64,2,2,0,97.0


Preview rows retained: 66


,iso3,country,year,gender__SG.TIM.UWRK.FE,gender__SG.TIM.UWRK.MA,join__JI.EMP.TOTL.SP.FE.ZS,join__JI.EMP.TOTL.SP.MA.ZS,gender__SL.TLF.CACT.FE.ZS,gender__SL.TLF.CACT.MA.ZS,wdi__EG.ELC.ACCS.ZS,wdi__SH.H2O.BASW.ZS,wdi__EG.CFT.ACCS.ZS,wdi__NY.GDP.PCAP.PP.KD,wdi__SP.DYN.TFRT.IN,wdi__SP.URB.TOTL.IN.ZS
0,ALB,Albania,2011,21.73611,3.47222,0.512444,0.654719,52.561,67.629,99.7,91.810884,68.6,12153.063659,1.677,53.291738
1,DZA,Algeria,2012,21.66667,3.75,<NA>,<NA>,14.821,69.989,98.8,90.954706,99.4,14697.539613,2.996,68.342126
2,ARG,Argentina,2010,17.5,6.59722,<NA>,<NA>,47.689,73.314,98.8,98.434052,99.2,28056.257021,2.411,90.993005
3,ARG,Argentina,2005,16.73611,6.11111,<NA>,<NA>,49.098,74.628,96.7,97.421185,98.0,23262.407579,2.426,90.123374
4,ARM,Armenia,2008,21.65833,4.37083,<NA>,<NA>,52.96,69.672,99.2,97.537793,94.3,11727.482127,1.4,64.384093


In [16]:
# Requires analysis_preview from the previous cells.
COLUMNS = {
    "female_time": "gender__SG.TIM.UWRK.FE",
    "male_time": "gender__SG.TIM.UWRK.MA",
    "female_lfpr": "gender__SL.TLF.CACT.FE.ZS",
    "male_lfpr": "gender__SL.TLF.CACT.MA.ZS",
    "female_employment": "join__JI.EMP.TOTL.SP.FE.ZS",
    "male_employment": "join__JI.EMP.TOTL.SP.MA.ZS",
    "electricity": "wdi__EG.ELC.ACCS.ZS",
    "water": "wdi__SH.H2O.BASW.ZS",
    "clean_cooking": "wdi__EG.CFT.ACCS.ZS",
    "gdp_per_capita": "wdi__NY.GDP.PCAP.PP.KD",
    "fertility": "wdi__SP.DYN.TFRT.IN",
    "urban_population": "wdi__SP.URB.TOTL.IN.ZS",
}

missing_parts = []

for label, column in COLUMNS.items():
    missing = analysis_preview.loc[
        analysis_preview[column].isna(),
        ["iso3", "country", "year"],
    ].copy()

    missing["indicator"] = label
    missing_parts.append(missing)

missing_records = pd.concat(missing_parts, ignore_index=True)

if missing_records.empty:
    print("No missing values in the selected indicators.")
else:
    for indicator, group in missing_records.groupby("indicator"):
        print(f"\nMissing observations: {indicator}")
        display(
            group[["iso3", "country", "year"]]
            .sort_values(["country", "year"])
            .reset_index(drop=True)
        )


Missing observations: clean_cooking


,iso3,country,year
0,AUS,Australia,2024
1,BGR,Bulgaria,2002
2,BGR,Bulgaria,2010
3,BGR,Bulgaria,2023
4,HKG,"Hong Kong SAR, China",2002
5,HKG,"Hong Kong SAR, China",2013



Missing observations: female_employment


,iso3,country,year
0,DZA,Algeria,2012
1,ARG,Argentina,2005
2,ARG,Argentina,2010
3,ARM,Armenia,2004
4,ARM,Armenia,2008
5,AUS,Australia,2006
6,AUS,Australia,2021
7,AUS,Australia,2024
8,AUT,Austria,2009
9,AUT,Austria,2022



Missing observations: gdp_per_capita


,iso3,country,year
0,CUB,Cuba,2001
1,CUB,Cuba,2016



Missing observations: male_employment


,iso3,country,year
0,DZA,Algeria,2012
1,ARG,Argentina,2005
2,ARG,Argentina,2010
3,ARM,Armenia,2004
4,ARM,Armenia,2008
5,AUS,Australia,2006
6,AUS,Australia,2021
7,AUS,Australia,2024
8,AUT,Austria,2009
9,AUT,Austria,2022


In [17]:
# Check overlap between clean cooking and GDP missingness.
cooking_missing = analysis_preview[
    COLUMNS["clean_cooking"]
].isna()

gdp_missing = analysis_preview[
    COLUMNS["gdp_per_capita"]
].isna()

missing_overlap = pd.DataFrame([
    {
        "missing_pattern": "Clean cooking only",
        "observations": int((cooking_missing & ~gdp_missing).sum()),
    },
    {
        "missing_pattern": "GDP only",
        "observations": int((~cooking_missing & gdp_missing).sum()),
    },
    {
        "missing_pattern": "Both",
        "observations": int((cooking_missing & gdp_missing).sum()),
    },
    {
        "missing_pattern": "Neither",
        "observations": int((~cooking_missing & ~gdp_missing).sum()),
    },
])

display(missing_overlap)

# Check whether male and female JOIN values exist in the same rows.
female_available = analysis_preview[
    COLUMNS["female_employment"]
].notna()

male_available = analysis_preview[
    COLUMNS["male_employment"]
].notna()

join_pairing = pd.DataFrame([
    {
        "availability": "Both available",
        "observations": int((female_available & male_available).sum()),
    },
    {
        "availability": "Female only",
        "observations": int((female_available & ~male_available).sum()),
    },
    {
        "availability": "Male only",
        "observations": int((~female_available & male_available).sum()),
    },
    {
        "availability": "Neither available",
        "observations": int((~female_available & ~male_available).sum()),
    },
])

display(join_pairing)

,missing_pattern,observations
0,Clean cooking only,6
1,GDP only,2
2,Both,0
3,Neither,58


,availability,observations
0,Both available,33
1,Female only,0
2,Male only,0
3,Neither available,33


In [18]:
core_features = [
    "female_time",
    "male_time",
    "female_lfpr",
    "male_lfpr",
    "electricity",
    "water",
]

SAMPLE_SPECS = {
    "Core": core_features,
    "Extended": core_features + [
        "clean_cooking",
        "gdp_per_capita",
        "fertility",
        "urban_population",
    ],
    "JOIN supplement": [
        "female_time",
        "male_time",
        "female_employment",
        "male_employment",
    ],
}

sample_views = {}
sample_summary_rows = []
country_coverage_parts = []

for sample_name, features in SAMPLE_SPECS.items():
    required_columns = [COLUMNS[feature] for feature in features]

    complete = analysis_preview[required_columns].notna().all(axis=1)

    sample = analysis_preview.loc[
        complete,
        ["iso3", "country", "year"] + required_columns,
    ].copy()

    sample_views[sample_name] = sample

    country_counts = (
        sample.groupby(["iso3", "country"])
        .size()
        .rename("observations")
        .reset_index()
    )

    sample_summary_rows.append({
        "sample": sample_name,
        "observations": len(sample),
        "entities": sample["iso3"].nunique(),
        "retained_pct": round(
            len(sample) / len(analysis_preview) * 100, 1
        ),
        "first_year": sample["year"].min(),
        "last_year": sample["year"].max(),
        "entities_with_2plus_observations": int(
            country_counts["observations"].ge(2).sum()
        ),
    })

    country_counts["sample"] = sample_name
    country_coverage_parts.append(country_counts)

sample_summary = pd.DataFrame(sample_summary_rows)

country_sample_counts = (
    pd.concat(country_coverage_parts, ignore_index=True)
    .pivot(
        index=["iso3", "country"],
        columns="sample",
        values="observations",
    )
    .reindex(columns=list(SAMPLE_SPECS))
    .fillna(0)
    .astype(int)
    .reset_index()
)

print("Sample size by indicator combination")
display(sample_summary)

print("Observations per entity in each sample")
display(country_sample_counts)

print(f"Original anchor rows retained: {len(analysis_preview)}")

Sample size by indicator combination


,sample,observations,entities,retained_pct,first_year,last_year,entities_with_2plus_observations
0,Core,66,27,100.0,2000,2024,19
1,Extended,58,24,87.9,2000,2023,16
2,JOIN supplement,33,14,50.0,2000,2021,5


Observations per entity in each sample


sample,iso3,country,Core,Extended,JOIN supplement
0,ALB,Albania,1,1,1
1,ARG,Argentina,2,2,0
2,ARM,Armenia,2,2,0
3,AUS,Australia,3,2,0
4,AUT,Austria,2,2,0
5,AZE,Azerbaijan,1,1,0
6,BEL,Belgium,3,3,1
7,BGR,Bulgaria,3,0,2
8,BLR,Belarus,1,1,0
9,BOL,Bolivia,1,1,1


Original anchor rows retained: 66


In [19]:
import numpy as np
import pandas as pd

# Requires analysis_preview and COLUMNS from the previous cells.
eda_data = analysis_preview.copy()

percentage_features = [
    "female_time",
    "male_time",
    "female_lfpr",
    "male_lfpr",
    "electricity",
    "water",
    "clean_cooking",
    "urban_population",
]

quality_rows = []
flagged_parts = []

for feature, column in COLUMNS.items():
    values = eda_data[column]
    observed = values.notna()

    finite = pd.Series(
        np.isfinite(values.to_numpy(dtype=float, na_value=np.nan)),
        index=values.index,
    )
    nonfinite = observed & ~finite

    if feature in percentage_features:
        outside_range = (
            observed & ~values.between(0, 100).fillna(False)
        )
        rule = "Expected range: 0 to 100"
    elif feature == "gdp_per_capita":
        outside_range = values.le(0).fillna(False)
        rule = "Expected value: greater than 0"
    elif feature == "fertility":
        outside_range = values.lt(0).fillna(False)
        rule = "Expected value: nonnegative"
    else:
        outside_range = pd.Series(False, index=values.index)
        rule = "Scale verification pending"

    flagged = nonfinite | outside_range

    quality_rows.append({
        "feature": feature,
        "nonmissing": int(observed.sum()),
        "missing": int((~observed).sum()),
        "minimum": values.min(),
        "maximum": values.max(),
        "flagged": int(flagged.sum()),
        "rule": rule,
    })

    if flagged.any():
        details = eda_data.loc[
            flagged, ["iso3", "country", "year", column]
        ].copy()
        details = details.rename(columns={column: "value"})
        details["feature"] = feature
        flagged_parts.append(details)

quality_report = pd.DataFrame(quality_rows)
display(quality_report)

if flagged_parts:
    flagged_values = pd.concat(flagged_parts, ignore_index=True)
    display(flagged_values)
else:
    print("No violations found under the checked rules.")

,feature,nonmissing,missing,minimum,maximum,flagged,rule
0,female_time,66,0,10.833330,25.416670,0,Expected range: 0 to 100
1,male_time,66,0,1.250000,12.638890,0,Expected range: 0 to 100
2,female_lfpr,66,0,14.821000,75.325000,0,Expected range: 0 to 100
3,male_lfpr,66,0,56.075000,86.653000,0,Expected range: 0 to 100
4,female_employment,33,33,0.180022,0.693703,0,Scale verification pending
5,male_employment,33,33,0.545979,0.826900,0,Scale verification pending
6,electricity,66,0,14.200000,100.000000,0,Expected range: 0 to 100
7,water,66,0,58.306111,100.000000,0,Expected range: 0 to 100
8,clean_cooking,60,6,5.900000,100.000000,0,Expected range: 0 to 100
9,gdp_per_capita,64,2,2491.309300,65694.459150,0,Expected value: greater than 0


No violations found under the checked rules.


In [20]:
# Preserve all original columns and add derived variables.
eda_data["female_unpaid_hours"] = (
    eda_data[COLUMNS["female_time"]] * 24 / 100
)

eda_data["male_unpaid_hours"] = (
    eda_data[COLUMNS["male_time"]] * 24 / 100
)

# Positive values mean women spend more time on unpaid work.
eda_data["time_gap_hours"] = (
    eda_data["female_unpaid_hours"]
    - eda_data["male_unpaid_hours"]
)

# Positive values mean male labor force participation is higher.
eda_data["lfpr_gap_pp"] = (
    eda_data[COLUMNS["male_lfpr"]]
    - eda_data[COLUMNS["female_lfpr"]]
)

display(
    eda_data[
        [
            "iso3",
            "country",
            "year",
            "female_unpaid_hours",
            "male_unpaid_hours",
            "time_gap_hours",
            "lfpr_gap_pp",
        ]
    ].head(10).round(3)
)

print(f"Rows retained: {len(eda_data)}")

,iso3,country,year,female_unpaid_hours,male_unpaid_hours,time_gap_hours,lfpr_gap_pp
0,ALB,Albania,2011,5.217,0.833,4.383,15.068
1,DZA,Algeria,2012,5.2,0.9,4.3,55.168
2,ARG,Argentina,2010,4.2,1.583,2.617,25.625
3,ARG,Argentina,2005,4.017,1.467,2.55,25.53
4,ARM,Armenia,2008,5.198,1.049,4.149,16.712
5,ARM,Armenia,2004,5.767,1.1,4.667,16.511
6,AUS,Australia,2024,4.3,3.033,1.267,8.271
7,AUS,Australia,2021,4.109,2.683,1.425,9.411
8,AUS,Australia,2006,5.0,2.733,2.267,14.748
9,AUT,Austria,2022,3.788,2.07,1.718,10.637


Rows retained: 66


In [21]:
summary_columns = {
    "female_unpaid_hours": "female_unpaid_hours",
    "male_unpaid_hours": "male_unpaid_hours",
    "time_gap_hours": "time_gap_hours",
    "female_lfpr_pct": COLUMNS["female_lfpr"],
    "male_lfpr_pct": COLUMNS["male_lfpr"],
    "lfpr_gap_pp": "lfpr_gap_pp",
    "electricity_pct": COLUMNS["electricity"],
    "basic_water_pct": COLUMNS["water"],
    "clean_cooking_pct": COLUMNS["clean_cooking"],
    "gdp_per_capita_ppp": COLUMNS["gdp_per_capita"],
    "fertility_rate": COLUMNS["fertility"],
    "urban_population_pct": COLUMNS["urban_population"],
}

latest_per_entity = (
    eda_data
    .sort_values(["iso3", "year"])
    .drop_duplicates(subset="iso3", keep="last")
    .copy()
)

def summarize_sample(frame):
    selected = frame[list(summary_columns.values())].copy()
    selected.columns = list(summary_columns.keys())

    summary = selected.describe(
        percentiles=[0.25, 0.50, 0.75]
    ).T

    summary["missing"] = selected.isna().sum()

    return summary.rename(columns={
        "count": "nonmissing",
        "50%": "median",
    })


all_observations_summary = summarize_sample(eda_data)
latest_observations_summary = summarize_sample(latest_per_entity)

print("All country-year observations")
display(all_observations_summary.round(3))

print("Latest observed time-use year per entity")
display(latest_observations_summary.round(3))

print(
    f"Latest-observation sample: {len(latest_per_entity)} entities; "
    f"years {latest_per_entity['year'].min()}–"
    f"{latest_per_entity['year'].max()}"
)

print("Year distribution of the latest-observation sample")
display(
    latest_per_entity.groupby("year")
    .size()
    .rename("entities")
    .to_frame()
)

All country-year observations


,nonmissing,mean,std,min,25%,median,75%,max,missing
female_unpaid_hours,66.0,4.248,0.824,2.6,3.76,4.222,4.873,6.1,0
male_unpaid_hours,66.0,1.76,0.666,0.3,1.246,1.681,2.371,3.033,0
time_gap_hours,66.0,2.489,0.913,1.0,1.655,2.418,2.97,4.783,0
female_lfpr_pct,66.0,52.628,8.824,14.821,48.126,53.239,56.532,75.325,0
male_lfpr_pct,66.0,72.782,5.98,56.075,69.758,73.452,77.05,86.653,0
lfpr_gap_pp,66.0,20.154,9.346,8.271,11.986,20.053,25.801,55.168,0
electricity_pct,66.0,95.871,12.539,14.2,97.425,99.2,100.0,100.0,0
basic_water_pct,66.0,94.205,7.631,58.306,90.19,97.363,99.401,100.0,0
clean_cooking_pct,60.0,89.468,17.817,5.9,90.775,94.4,99.85,100.0,6
gdp_per_capita_ppp,64.0,25807.539,18490.434,2491.309,12833.04,17862.399,34869.797,65694.459,2


Latest observed time-use year per entity


,nonmissing,mean,std,min,25%,median,75%,max,missing
female_unpaid_hours,27.0,4.309,0.867,2.6,3.769,4.234,5.118,6.1,0
male_unpaid_hours,27.0,1.773,0.799,0.3,1.176,1.583,2.478,3.033,0
time_gap_hours,27.0,2.535,1.038,1.05,1.759,2.317,2.882,4.783,0
female_lfpr_pct,27.0,52.729,11.845,14.821,50.656,53.869,58.749,75.325,0
male_lfpr_pct,27.0,71.632,5.785,59.306,68.061,70.918,74.556,86.653,0
lfpr_gap_pp,27.0,18.903,11.398,8.271,11.254,14.977,23.522,55.168,0
electricity_pct,27.0,93.404,18.649,14.2,98.45,99.7,100.0,100.0,0
basic_water_pct,27.0,93.174,10.338,58.306,91.383,97.538,99.408,100.0,0
clean_cooking_pct,24.0,85.688,24.297,5.9,86.45,94.4,99.575,100.0,3
gdp_per_capita_ppp,26.0,26917.824,19982.448,2491.309,12656.702,18258.533,32322.135,65694.459,1


Latest-observation sample: 27 entities; years 2001–2024
Year distribution of the latest-observation sample


,entities
year,
2001,1
2004,1
2008,2
2009,1
2010,2
2011,1
2012,1
2013,1
2014,2


In [22]:
from pathlib import Path
import pandas as pd

ROOT = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\all_sources"
)

catalog_parts = []

for catalog_path in sorted(ROOT.glob("*/_indicators.csv")):
    catalog = pd.read_csv(
        catalog_path,
        dtype="string",
        keep_default_na=False,
    )

    catalog["database_folder"] = catalog_path.parent.name
    catalog_parts.append(catalog)

if not catalog_parts:
    raise FileNotFoundError("No indicator catalogs found.")

all_catalogs = pd.concat(catalog_parts, ignore_index=True)

search_text = (
    all_catalogs["name"].fillna("")
    + " "
    + all_catalogs["definition"].fillna("")
).str.lower()

time_terms = search_text.str.contains(
    r"\btime\b|\bhours?\b|\bminutes?\b",
    regex=True,
)

work_terms = search_text.str.contains(
    r"unpaid|domestic|caregiving|care work|housework|household chores",
    regex=True,
)

known_time_codes = all_catalogs["id"].isin([
    "SG.TIM.UWRK.FE",
    "SG.TIM.UWRK.MA",
])

time_candidates = (
    all_catalogs.loc[
        (time_terms & work_terms) | known_time_codes
    ]
    .copy()
    .reset_index(drop=True)
)

print(f"Database catalogs scanned: {len(catalog_parts)}")
print(f"Candidate indicators found: {len(time_candidates)}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 120,
):
    display(
        time_candidates[
            ["database_folder", "id", "name"]
        ]
    )

Database catalogs scanned: 7
Candidate indicators found: 53


,database_folder,id,name
0,002_WDI,FM.AST.DOMS.CN,Net domestic credit (current LCU)
1,002_WDI,FS.AST.DOMS.GD.ZS,Domestic credit provided by financial sector (% of GDP)
2,002_WDI,FS.AST.PRVT.GD.ZS,Domestic credit to private sector (% of GDP)
3,002_WDI,GC.AST.TOTL.CN,Net acquisition of financial assets (current LCU)
4,002_WDI,GC.DOD.TOTL.CN,"Central government debt, total (current LCU)"
5,002_WDI,GC.LBL.TOTL.CN,"Net incurrence of liabilities, total (current LCU)"
6,002_WDI,GC.XPN.INTP.CN,Interest payments (current LCU)
7,002_WDI,IC.BRE.BE.P3,B-READY: Business Entry Pillar 3: Operational Efficiency of Business Entry
8,002_WDI,NE.GDI.FPRV.CN,"Gross fixed capital formation, private sector (current LCU)"
9,002_WDI,NY.GDP.MKTP.CD,GDP (current US$)


In [23]:
time_candidate_data = {}
coverage_rows = []

for row in time_candidates.itertuples(index=False):
    path = ROOT / row.database_folder / f"{row.id}.csv.gz"

    result = {
        "database_folder": row.database_folder,
        "indicator": row.id,
        "name": row.name,
        "file_exists": path.is_file(),
    }

    if not path.is_file():
        coverage_rows.append(result)
        continue

    raw = pd.read_csv(
        path,
        compression="gzip",
        dtype="string",
        keep_default_na=False,
    )

    value_text = raw["value"].str.strip()
    year_text = raw["date"].str.strip()

    values = pd.to_numeric(
        value_text.replace("", pd.NA), errors="coerce"
    )
    years = pd.to_numeric(
        year_text.replace("", pd.NA), errors="coerce"
    )

    valid = values.notna()

    result.update({
        "file_rows": len(raw),
        "numeric_observations": int(valid.sum()),
        "entity_ids": raw.loc[
            valid, "country_id"
        ].replace("", pd.NA).nunique(),
        "first_year": years.loc[valid].min(),
        "last_year": years.loc[valid].max(),
        "minimum": values.min(),
        "maximum": values.max(),
        "unparsed_values": int(
            (value_text.ne("") & values.isna()).sum()
        ),
        "unparsed_years": int(
            (year_text.ne("") & years.isna()).sum()
        ),
    })

    time_candidate_data[(row.database_folder, row.id)] = raw
    coverage_rows.append(result)

time_source_coverage = pd.DataFrame(coverage_rows)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
):
    display(time_source_coverage)

,database_folder,indicator,name,file_exists,file_rows,numeric_observations,entity_ids,first_year,last_year,minimum,maximum,unparsed_values,unparsed_years
0,002_WDI,FM.AST.DOMS.CN,Net domestic credit (current LCU),True,8948,8948,186,1960,2025,-5.423764e+13,1.210829e+16,0,0
1,002_WDI,FS.AST.DOMS.GD.ZS,Domestic credit provided by financial sector (...,True,1700,1700,78,1960,2025,-9.313642e+00,3.831251e+02,0,0
2,002_WDI,FS.AST.PRVT.GD.ZS,Domestic credit to private sector (% of GDP),True,9883,9883,233,1960,2025,7.725797e-03,3.010189e+02,0,0
3,002_WDI,GC.AST.TOTL.CN,Net acquisition of financial assets (current LCU),True,2515,2515,146,1970,2024,-5.321200e+13,9.976233e+13,0,0
4,002_WDI,GC.DOD.TOTL.CN,"Central government debt, total (current LCU)",True,1650,1650,109,1970,2024,-1.251080e+09,1.693690e+15,0,0
5,002_WDI,GC.LBL.TOTL.CN,"Net incurrence of liabilities, total (current ...",True,3005,3005,149,1972,2024,-4.626897e+13,1.293119e+14,0,0
6,002_WDI,GC.XPN.INTP.CN,Interest payments (current LCU),True,4521,4521,161,1972,2024,2.768767e-01,9.378190e+13,0,0
7,002_WDI,IC.BRE.BE.P3,B-READY: Business Entry Pillar 3: Operational ...,True,50,50,50,2024,2024,9.000000e+00,1.000000e+02,0,0
8,002_WDI,NE.GDI.FPRV.CN,"Gross fixed capital formation, private sector ...",True,3088,3088,109,1960,2025,-6.127600e+06,6.504904e+14,0,0
9,002_WDI,NY.GDP.MKTP.CD,GDP (current US$),True,14745,14745,261,1960,2025,2.585956e+06,1.183502e+14,0,0


In [24]:
def extract_valid_time_keys(raw):
    values = pd.to_numeric(
        raw["value"].str.strip().replace("", pd.NA),
        errors="coerce",
    )

    year_text = raw["date"].str.strip()
    annual_year = year_text.str.fullmatch(r"\d{4}", na=False)
    iso3 = raw["iso3"].str.strip()

    # Use only explicitly supplied three-letter identifiers.
    usable = values.notna() & annual_year & iso3.ne("")

    keys = pd.DataFrame({
        "iso3": iso3.loc[usable],
        "year": year_text.loc[usable].astype(int),
    })

    diagnostics = {
        "numeric_rows_without_iso3": int(
            (values.notna() & iso3.eq("")).sum()
        ),
        "numeric_rows_with_nonannual_date": int(
            (values.notna() & ~annual_year).sum()
        ),
        "duplicate_keys": int(keys.duplicated().sum()),
    }

    return keys.drop_duplicates(), diagnostics


comparison_rows = []
new_time_keys = {}

for code in ["SG.TIM.UWRK.FE", "SG.TIM.UWRK.MA"]:
    baseline_key = ("014_GDS", code)

    if baseline_key not in time_candidate_data:
        raise ValueError(f"Missing baseline file: {baseline_key}")

    baseline, _ = extract_valid_time_keys(
        time_candidate_data[baseline_key]
    )
    baseline_entities = set(baseline["iso3"])

    for (folder, indicator), raw in time_candidate_data.items():
        if indicator != code:
            continue

        candidate_keys, diagnostics = extract_valid_time_keys(raw)

        comparison = candidate_keys.merge(
            baseline,
            on=["iso3", "year"],
            how="left",
            indicator=True,
            validate="one_to_one",
        )

        added = comparison.loc[
            comparison["_merge"].eq("left_only"),
            ["iso3", "year"],
        ].copy()

        new_time_keys[(folder, code)] = added

        comparison_rows.append({
            "database_folder": folder,
            "indicator": code,
            "comparable_keys": len(candidate_keys),
            "additional_country_years": len(added),
            "additional_entities": len(
                set(candidate_keys["iso3"]) - baseline_entities
            ),
            **diagnostics,
        })

time_source_comparison = pd.DataFrame(comparison_rows)
display(time_source_comparison)

for (folder, code), added in new_time_keys.items():
    if not added.empty:
        print(f"\nAdditional observations: {folder} / {code}")
        display(added.sort_values(["iso3", "year"]))

,database_folder,indicator,comparable_keys,additional_country_years,additional_entities,numeric_rows_without_iso3,numeric_rows_with_nonannual_date,duplicate_keys
0,002_WDI,SG.TIM.UWRK.FE,66,0,0,0,0,0
1,014_GDS,SG.TIM.UWRK.FE,66,0,0,0,0,0
2,002_WDI,SG.TIM.UWRK.MA,66,0,0,0,0,0
3,014_GDS,SG.TIM.UWRK.MA,66,0,0,0,0,0


In [25]:
from pathlib import Path
import re
import pandas as pd

ROOT = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\all_sources"
)

folders = [
    path for path in ROOT.glob("046_*")
    if path.is_dir()
]

if len(folders) != 1:
    raise RuntimeError(
        f"Expected one source-46 folder, found {len(folders)}."
    )

sdg_folder = folders[0]

sdg_catalog = pd.read_csv(
    sdg_folder / "_indicators.csv",
    dtype="string",
    keep_default_na=False,
)

def indicator_path(code):
    filename = re.sub(r"[^A-Za-z0-9._-]", "_", code)
    return sdg_folder / f"{filename}.csv.gz"


sdg_catalog["file_exists"] = sdg_catalog["id"].map(
    lambda code: indicator_path(code).is_file()
)

print(f"Database folder: {sdg_folder.name}")
print(f"Catalog indicators: {len(sdg_catalog)}")
print(f"Files present: {sdg_catalog['file_exists'].sum()}")

missing_files = sdg_catalog.loc[
    ~sdg_catalog["file_exists"], ["id", "name"]
]

if not missing_files.empty:
    print("\nMissing files")
    display(missing_files)

log_path = sdg_folder / "_log.csv"

if log_path.is_file():
    log = pd.read_csv(
        log_path, dtype="string", keep_default_na=False
    )
    latest_log = log.drop_duplicates("indicator", keep="last")

    print("\nLatest download status per indicator")
    display(latest_log["status"].value_counts().to_frame("indicators"))

    errors = latest_log.loc[latest_log["status"].eq("error")]
    if not errors.empty:
        display(errors)

names = sdg_catalog["name"].str.lower()

time_measure = names.str.contains(
    r"\btime\b|\bhours?\b|\bminutes?\b", regex=True
)

care_activity = names.str.contains(
    r"unpaid|domestic work|domestic chores|care work|"
    r"caregiving|housework|household chores",
    regex=True,
)

known_codes = sdg_catalog["id"].isin([
    "SG.TIM.UWRK.FE",
    "SG.TIM.UWRK.MA",
])

sdg_time_candidates = sdg_catalog.loc[
    (time_measure & care_activity) | known_codes
].copy()

print(f"\nTime-use candidates: {len(sdg_time_candidates)}")

with pd.option_context(
    "display.max_rows", None,
    "display.max_colwidth", 150,
):
    display(
        sdg_time_candidates[["id", "name", "file_exists"]]
    )

for row in sdg_time_candidates.itertuples(index=False):
    print(f"\nIndicator: {row.id}")
    print(f"Name: {row.name}")
    print(f"Definition: {row.definition or '[Not provided]'}")

Database folder: 046_SDG
Catalog indicators: 408
Files present: 408

Latest download status per indicator


,indicators
status,
ok,404
empty,4



Time-use candidates: 2


,id,name,file_exists
248,SG.TIM.UWRK.FE,"Proportion of time spent on unpaid domestic and care work, female (% of 24 hour day)",True
249,SG.TIM.UWRK.MA,"Proportion of time spent on unpaid domestic and care work, male (% of 24 hour day)",True



Indicator: SG.TIM.UWRK.FE
Name: Proportion of time spent on unpaid domestic and care work, female (% of 24 hour day)
Definition: The average time women spend on household provision of services for own consumption. Data are expressed as a proportion of time in a day. Domestic and care work includes food preparation, dishwashing, cleaning and upkeep of a dwelling, laundry, ironing, gardening, caring for pets, shopping, installation, servicing and repair of personal and household goods, childcare, and care of the sick, elderly or disabled household members, among others.

Indicator: SG.TIM.UWRK.MA
Name: Proportion of time spent on unpaid domestic and care work, male (% of 24 hour day) 
Definition: The average time men spend on household provision of services for own consumption.  Data are expressed as a proportion of time in a day. Domestic and care work includes food preparation, dishwashing, cleaning and upkeep of a dwelling, laundry, ironing, gardening, caring for pets, shopping, inst

In [26]:
comparison_rows = []
sdg_additional_keys = {}

for code in ["SG.TIM.UWRK.FE", "SG.TIM.UWRK.MA"]:
    if code not in set(sdg_catalog["id"]):
        print(f"Not listed in source 46: {code}")
        continue

    path = indicator_path(code)

    if not path.is_file():
        print(f"Download missing; comparison unavailable: {code}")
        continue

    sdg_raw = pd.read_csv(
        path, dtype="string", keep_default_na=False
    )
    baseline_raw = pd.read_csv(
        ROOT / "014_GDS" / f"{code}.csv.gz",
        dtype="string",
        keep_default_na=False,
    )

    sdg_keys, diagnostics = extract_valid_time_keys(sdg_raw)
    baseline_keys, _ = extract_valid_time_keys(baseline_raw)

    compared = sdg_keys.merge(
        baseline_keys,
        on=["iso3", "year"],
        how="outer",
        indicator=True,
        validate="one_to_one",
    )

    additional = compared.loc[
        compared["_merge"].eq("left_only"),
        ["iso3", "year"],
    ].copy()

    sdg_additional_keys[code] = additional

    comparison_rows.append({
        "indicator": code,
        "sdg_country_years": len(sdg_keys),
        "sdg_entities": sdg_keys["iso3"].nunique(),
        "additional_country_years": len(additional),
        "additional_entities": len(
            set(sdg_keys["iso3"]) - set(baseline_keys["iso3"])
        ),
        "baseline_only_keys": int(
            compared["_merge"].eq("right_only").sum()
        ),
        **diagnostics,
    })

    if not additional.empty:
        print(f"\nAdditional observations: {code}")
        display(additional.sort_values(["iso3", "year"]))

display(pd.DataFrame(comparison_rows))

,indicator,sdg_country_years,sdg_entities,additional_country_years,additional_entities,baseline_only_keys,numeric_rows_without_iso3,numeric_rows_with_nonannual_date,duplicate_keys
0,SG.TIM.UWRK.FE,0,0,0,0,66,177,0,0
1,SG.TIM.UWRK.MA,0,0,0,0,66,177,0,0


In [27]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
    r"\Graduate_Dataset\all_sources"
)

TIME_SOURCES = {
    "gender": ROOT / "014_GDS",
    "sdg": ROOT / "046_SDG",
}

TIME_INDICATORS = {
    "female": "SG.TIM.UWRK.FE",
    "male": "SG.TIM.UWRK.MA",
}

time_parts = []

for source, folder in TIME_SOURCES.items():
    for sex, code in TIME_INDICATORS.items():
        raw = pd.read_csv(
            folder / f"{code}.csv.gz",
            dtype="string",
            keep_default_na=False,
        )

        reported_iso3 = raw["iso3"].str.strip()
        country_id = raw["country_id"].str.strip()

        if source == "sdg":
            conflict = (
                reported_iso3.ne("")
                & reported_iso3.ne(country_id)
            )
            if conflict.any():
                raise ValueError(f"Conflicting SDG codes: {code}")

            iso3 = reported_iso3.mask(
                reported_iso3.eq(""), country_id
            )
        else:
            iso3 = reported_iso3

        if not iso3.str.fullmatch(r"[A-Z]{3}", na=False).all():
            raise ValueError(f"Invalid country codes: {source}, {code}")

        year_text = raw["date"].str.strip()
        if not year_text.str.fullmatch(r"\d{4}", na=False).all():
            raise ValueError(f"Invalid annual dates: {source}, {code}")

        values = pd.to_numeric(
            raw["value"].str.strip().replace("", pd.NA),
            errors="raise",
        )

        part = pd.DataFrame({
            "source": source,
            "indicator": code,
            "sex": sex,
            "iso3": iso3,
            "country": raw["country"],
            "year": year_text.astype(int),
            "time_pct_day": values,
        }).dropna(subset=["time_pct_day"])

        if not part["time_pct_day"].between(0, 100).all():
            raise ValueError(f"Time values outside 0–100: {source}, {code}")

        if part.duplicated(["iso3", "year"]).any():
            raise ValueError(f"Duplicate country-year keys: {source}, {code}")

        part["unpaid_hours"] = part["time_pct_day"] * 24 / 100
        time_parts.append(part)

time_long = pd.concat(time_parts, ignore_index=True)

time_coverage = (
    time_long.groupby(["source", "sex"])
    .agg(
        observations=("time_pct_day", "size"),
        entities=("iso3", "nunique"),
        first_year=("year", "min"),
        last_year=("year", "max"),
    )
    .reset_index()
)

display(time_coverage)

,source,sex,observations,entities,first_year,last_year
0,gender,female,66,27,2000,2024
1,gender,male,66,27,2000,2024
2,sdg,female,177,89,2000,2019
3,sdg,male,177,89,2000,2019


In [28]:
comparison_keys = ["iso3", "year", "sex"]

gender_time = time_long.loc[
    time_long["source"].eq("gender"),
    comparison_keys + ["country", "time_pct_day"],
]

sdg_time = time_long.loc[
    time_long["source"].eq("sdg"),
    comparison_keys + ["country", "time_pct_day"],
]

source_comparison = gender_time.merge(
    sdg_time,
    on=comparison_keys,
    how="outer",
    suffixes=("_gender", "_sdg"),
    indicator=True,
    validate="one_to_one",
)

source_comparison["country"] = (
    source_comparison["country_gender"]
    .combine_first(source_comparison["country_sdg"])
)

both = source_comparison["_merge"].eq("both")
overlap = source_comparison.loc[both].copy()

overlap["difference_pp"] = (
    overlap["time_pct_day_sdg"]
    - overlap["time_pct_day_gender"]
)
overlap["difference_hours"] = overlap["difference_pp"] * 24 / 100

overlap["value_conflict"] = ~np.isclose(
    overlap["time_pct_day_gender"].to_numpy(dtype=float),
    overlap["time_pct_day_sdg"].to_numpy(dtype=float),
    rtol=0,
    atol=1e-5,
)

source_conflicts = (
    overlap.loc[
        overlap["value_conflict"],
        [
            "iso3", "country", "year", "sex",
            "time_pct_day_gender", "time_pct_day_sdg",
            "difference_pp", "difference_hours",
        ],
    ]
    .sort_values(["country", "year", "sex"])
    .reset_index(drop=True)
)

coverage_comparison = pd.crosstab(
    source_comparison["sex"],
    source_comparison["_merge"],
).rename(columns={
    "left_only": "gender_only",
    "right_only": "sdg_only",
    "both": "shared",
})

print("Country-year coverage by sex")
display(coverage_comparison)

print("Conflicting values: SDG minus Gender")
display(source_conflicts.round(5))

print(
    "Country-year pairs with at least one sex-specific conflict:",
    len(source_conflicts[["iso3", "year"]].drop_duplicates()),
)

Country-year coverage by sex


_merge,gender_only,sdg_only,shared
sex,,,
female,23,134,43
male,23,134,43


Conflicting values: SDG minus Gender


,iso3,country,year,sex,time_pct_day_gender,time_pct_day_sdg,difference_pp,difference_hours
0,BRA,Brazil,2017,female,11.74614,11.60814,-0.138,-0.03312
1,BRA,Brazil,2017,male,5.17493,5.13276,-0.04217,-0.01012
2,CAN,Canada,2005,female,15.83333,16.25,0.41667,0.1
3,CAN,Canada,2016,female,14.16667,14.58333,0.41666,0.1
4,CAN,Canada,2016,male,10.0,9.58333,-0.41667,-0.1
5,CHN,China,2008,male,6.18056,6.25,0.06944,0.01667
6,COL,Colombia,2017,female,16.87229,5.02051,-11.85178,-2.84443
7,COL,Colombia,2017,male,4.59313,2.92763,-1.6655,-0.39972


Country-year pairs with at least one sex-specific conflict: 5


In [29]:
conflict_keys = (
    source_conflicts[["iso3", "year"]]
    .drop_duplicates()
    .assign(cross_source_conflict=True)
)

time_samples = {}
sample_rows = []

for source in TIME_SOURCES:
    source_data = time_long.loc[
        time_long["source"].eq(source)
    ]

    female = source_data.loc[
        source_data["sex"].eq("female"),
        ["iso3", "country", "year", "unpaid_hours"],
    ].rename(columns={"unpaid_hours": "female_unpaid_hours"})

    male = source_data.loc[
        source_data["sex"].eq("male"),
        ["iso3", "year", "unpaid_hours"],
    ].rename(columns={"unpaid_hours": "male_unpaid_hours"})

    paired = female.merge(
        male,
        on=["iso3", "year"],
        how="outer",
        validate="one_to_one",
        indicator=True,
    )

    if not paired["_merge"].eq("both").all():
        display(paired.loc[paired["_merge"].ne("both")])
        raise ValueError(f"Unpaired observations in {source}")

    paired = paired.drop(columns="_merge")
    paired["source"] = source
    paired["time_gap_hours"] = (
        paired["female_unpaid_hours"]
        - paired["male_unpaid_hours"]
    )

    paired = paired.merge(
        conflict_keys,
        on=["iso3", "year"],
        how="left",
        validate="one_to_one",
    )
    paired["cross_source_conflict"] = (
        paired["cross_source_conflict"].eq(True)
    )

    time_samples[source] = paired

    sample_rows.append({
        "source": source,
        "paired_observations": len(paired),
        "entities": paired["iso3"].nunique(),
        "first_year": paired["year"].min(),
        "last_year": paired["year"].max(),
        "flagged_country_years": int(
            paired["cross_source_conflict"].sum()
        ),
    })

display(pd.DataFrame(sample_rows))

print("SDG paired sample preview")
display(time_samples["sdg"].head())

,source,paired_observations,entities,first_year,last_year,flagged_country_years
0,gender,66,27,2000,2024,5
1,sdg,177,89,2000,2019,5


SDG paired sample preview


,iso3,country,year,female_unpaid_hours,male_unpaid_hours,source,time_gap_hours,cross_source_conflict
0,ALB,Albania,2011,5.216666,0.833333,sdg,4.383334,False
1,ARG,Argentina,2005,4.016666,1.466666,sdg,2.55,False
2,ARG,Argentina,2010,4.2,1.583333,sdg,2.616667,False
3,ARG,Argentina,2013,5.620474,2.219664,sdg,3.40081,False
4,ARM,Armenia,2004,5.766667,1.099999,sdg,4.666668,False


In [30]:
# Requires ROOT, FOLDERS, indicator_dictionary, and time_samples.
import pandas as pd

mapping_raw = pd.read_csv(
    ROOT / "002_WDI" / "EG.ELC.ACCS.ZS.csv.gz",
    dtype="string",
    keep_default_na=False,
)

country_crosswalk = (
    mapping_raw[["country_id", "iso3"]]
    .apply(lambda column: column.str.strip())
    .query("country_id != '' and iso3 != ''")
    .drop_duplicates()
)

if (
    country_crosswalk["country_id"].duplicated().any()
    or country_crosswalk["iso3"].duplicated().any()
):
    raise ValueError("Country-code mapping is not one-to-one.")

id_to_iso3 = country_crosswalk.set_index("country_id")["iso3"].to_dict()
known_iso3 = set(country_crosswalk["iso3"])

target_iso3 = set().union(
    *(set(sample["iso3"]) for sample in time_samples.values())
)

unmapped_targets = target_iso3 - known_iso3

print(f"Target entities across both time samples: {len(target_iso3)}")
print(f"Target entities absent from crosswalk: {len(unmapped_targets)}")

if unmapped_targets:
    print(sorted(unmapped_targets))
    raise ValueError("Resolve missing country mappings before continuing.")

Target entities across both time samples: 89
Target entities absent from crosswalk: 0


In [31]:
covariate_specs = indicator_dictionary.loc[
    ~indicator_dictionary["indicator"].isin(TIME_INDICATORS.values())
].copy()

prepared_covariates = {}
mapping_audit_rows = []

for row in covariate_specs.itertuples(index=False):
    raw = pd.read_csv(
        FOLDERS[row.source] / f"{row.indicator}.csv.gz",
        dtype="string",
        keep_default_na=False,
    )

    country_id = raw["country_id"].str.strip()
    reported_iso3 = raw["iso3"].str.strip()

    # Resolve IDs using verified two-letter mappings or known
    # three-letter codes. Do not infer codes from string length.
    from_id = country_id.map(id_to_iso3)
    from_id = from_id.mask(country_id.isin(known_iso3), country_id)

    conflict = (
        reported_iso3.ne("")
        & from_id.notna()
        & reported_iso3.ne(from_id).fillna(False)
    )

    if conflict.any():
        display(raw.loc[conflict].head(10))
        raise ValueError(f"Country-code conflict: {row.indicator}")

    resolved_iso3 = reported_iso3.mask(
        reported_iso3.eq(""), from_id
    )
    keep = resolved_iso3.isin(target_iso3)

    mapping_audit_rows.append({
        "source": row.source,
        "indicator": row.indicator,
        "raw_rows": len(raw),
        "rows_in_target_entities": int(keep.sum()),
        "rows_without_resolved_code": int(
            (resolved_iso3.isna() | resolved_iso3.eq("").fillna(False)).sum()
        ),
    })

    selected = raw.loc[keep].copy()
    year_text = selected["date"].str.strip()

    if not year_text.str.fullmatch(r"\d{4}", na=False).all():
        raise ValueError(f"Nonannual dates: {row.indicator}")

    prepared = pd.DataFrame({
        "iso3": resolved_iso3.loc[keep],
        "year": year_text.astype(int),
        "value": pd.to_numeric(
            selected["value"].str.strip().replace("", pd.NA),
            errors="raise",
        ),
    })

    if prepared.duplicated(["iso3", "year"]).any():
        raise ValueError(f"Duplicate country-year keys: {row.indicator}")

    column = f"{row.source}__{row.indicator}"
    prepared_covariates[column] = prepared.rename(
        columns={"value": column}
    )

expanded_samples = {}
expanded_coverage_rows = []

for sample_name, anchor in time_samples.items():
    matched = anchor.copy()

    for column, prepared in prepared_covariates.items():
        matched = matched.merge(
            prepared,
            on=["iso3", "year"],
            how="left",
            validate="one_to_one",
        )

        available = matched[column].notna()

        expanded_coverage_rows.append({
            "time_source": sample_name,
            "indicator": column,
            "anchor_observations": len(anchor),
            "matched_observations": int(available.sum()),
            "missing_observations": int((~available).sum()),
            "matched_entities": matched.loc[
                available, "iso3"
            ].nunique(),
            "match_rate_pct": round(available.mean() * 100, 1),
        })

    if len(matched) != len(anchor):
        raise ValueError(f"Anchor size changed: {sample_name}")

    expanded_samples[sample_name] = matched

mapping_audit = pd.DataFrame(mapping_audit_rows)
expanded_coverage = pd.DataFrame(expanded_coverage_rows)

print("Country-code mapping diagnostics")
display(mapping_audit)

for sample_name in expanded_samples:
    print(f"\nSame-year coverage: {sample_name}")
    with pd.option_context("display.max_columns", None):
        display(
            expanded_coverage.loc[
                expanded_coverage["time_source"].eq(sample_name)
            ].reset_index(drop=True)
        )

print("\nAnchor rows retained")
for name, frame in expanded_samples.items():
    print(f"{name}: {len(frame)}")

Country-code mapping diagnostics


,source,indicator,raw_rows,rows_in_target_entities,rows_without_resolved_code
0,join,JI.EMP.TOTL.SP.FE.ZS,1639,1173,0
1,join,JI.EMP.TOTL.SP.MA.ZS,1639,1173,0
2,gender,SL.TLF.CACT.FE.ZS,8410,3201,144
3,gender,SL.TLF.CACT.MA.ZS,8410,3201,144
4,wdi,EG.ELC.ACCS.ZS,8097,2854,117
5,wdi,SH.H2O.BASW.ZS,6373,2205,100
6,wdi,EG.CFT.ACCS.ZS,5664,2064,96
7,wdi,NY.GDP.PCAP.PP.KD,8669,3159,144
8,wdi,SP.DYN.TFRT.IN,17128,5753,260
9,wdi,SP.URB.TOTL.IN.ZS,17424,5874,264



Same-year coverage: gender


,time_source,indicator,anchor_observations,matched_observations,missing_observations,matched_entities,match_rate_pct
0,gender,join__JI.EMP.TOTL.SP.FE.ZS,66,33,33,14,50.0
1,gender,join__JI.EMP.TOTL.SP.MA.ZS,66,33,33,14,50.0
2,gender,gender__SL.TLF.CACT.FE.ZS,66,66,0,27,100.0
3,gender,gender__SL.TLF.CACT.MA.ZS,66,66,0,27,100.0
4,gender,wdi__EG.ELC.ACCS.ZS,66,66,0,27,100.0
5,gender,wdi__SH.H2O.BASW.ZS,66,66,0,27,100.0
6,gender,wdi__EG.CFT.ACCS.ZS,66,60,6,25,90.9
7,gender,wdi__NY.GDP.PCAP.PP.KD,66,64,2,26,97.0
8,gender,wdi__SP.DYN.TFRT.IN,66,66,0,27,100.0
9,gender,wdi__SP.URB.TOTL.IN.ZS,66,66,0,27,100.0



Same-year coverage: sdg


,time_source,indicator,anchor_observations,matched_observations,missing_observations,matched_entities,match_rate_pct
0,sdg,join__JI.EMP.TOTL.SP.FE.ZS,177,100,77,59,56.5
1,sdg,join__JI.EMP.TOTL.SP.MA.ZS,177,100,77,59,56.5
2,sdg,gender__SL.TLF.CACT.FE.ZS,177,177,0,89,100.0
3,sdg,gender__SL.TLF.CACT.MA.ZS,177,177,0,89,100.0
4,sdg,wdi__EG.ELC.ACCS.ZS,177,177,0,89,100.0
5,sdg,wdi__SH.H2O.BASW.ZS,177,174,3,89,98.3
6,sdg,wdi__EG.CFT.ACCS.ZS,177,171,6,86,96.6
7,sdg,wdi__NY.GDP.PCAP.PP.KD,177,175,2,88,98.9
8,sdg,wdi__SP.DYN.TFRT.IN,177,177,0,89,100.0
9,sdg,wdi__SP.URB.TOTL.IN.ZS,177,177,0,89,100.0



Anchor rows retained
gender: 66
sdg: 177


In [32]:
core_columns = [
    "gender__SL.TLF.CACT.FE.ZS",
    "gender__SL.TLF.CACT.MA.ZS",
    "wdi__EG.ELC.ACCS.ZS",
    "wdi__SH.H2O.BASW.ZS",
]

ANALYSIS_COMBINATIONS = {
    "Core": core_columns,
    "Extended": core_columns + [
        "wdi__EG.CFT.ACCS.ZS",
        "wdi__NY.GDP.PCAP.PP.KD",
        "wdi__SP.DYN.TFRT.IN",
        "wdi__SP.URB.TOTL.IN.ZS",
    ],
    "JOIN supplement": [
        "join__JI.EMP.TOTL.SP.FE.ZS",
        "join__JI.EMP.TOTL.SP.MA.ZS",
    ],
}

combination_rows = []
expanded_sample_views = {}

for source, frame in expanded_samples.items():
    for combination, required in ANALYSIS_COMBINATIONS.items():
        for exclude_conflicts in [False, True]:
            complete = frame[required].notna().all(axis=1)

            if exclude_conflicts:
                complete = complete & ~frame["cross_source_conflict"]

            subset = frame.loc[complete].copy()
            entity_counts = subset.groupby("iso3").size()

            policy = (
                "Exclude flagged pairs"
                if exclude_conflicts
                else "Keep all pairs"
            )

            expanded_sample_views[(source, combination, policy)] = subset

            combination_rows.append({
                "time_source": source,
                "combination": combination,
                "conflict_policy": policy,
                "observations": len(subset),
                "entities": subset["iso3"].nunique(),
                "retained_pct_of_anchor": round(
                    len(subset) / len(frame) * 100, 1
                ),
                "first_year": subset["year"].min(),
                "last_year": subset["year"].max(),
                "entities_with_2plus_observations": int(
                    entity_counts.ge(2).sum()
                ),
                "largest_entity_share_pct": (
                    round(entity_counts.max() / len(subset) * 100, 1)
                    if len(subset) else float("nan")
                ),
            })

expanded_sample_summary = pd.DataFrame(combination_rows)

with pd.option_context(
    "display.max_columns", None,
    "display.max_rows", None,
):
    display(expanded_sample_summary)

,time_source,combination,conflict_policy,observations,entities,retained_pct_of_anchor,first_year,last_year,entities_with_2plus_observations,largest_entity_share_pct
0,gender,Core,Keep all pairs,66,27,100.0,2000,2024,19,16.7
1,gender,Core,Exclude flagged pairs,61,27,92.4,2000,2024,18,18.0
2,gender,Extended,Keep all pairs,58,24,87.9,2000,2023,16,19.0
3,gender,Extended,Exclude flagged pairs,53,24,80.3,2000,2023,15,20.8
4,gender,JOIN supplement,Keep all pairs,33,14,50.0,2000,2021,5,33.3
5,gender,JOIN supplement,Exclude flagged pairs,31,14,47.0,2000,2021,5,35.5
6,sdg,Core,Keep all pairs,174,89,98.3,2000,2019,44,8.6
7,sdg,Core,Exclude flagged pairs,169,89,95.5,2000,2019,41,8.9
8,sdg,Extended,Keep all pairs,166,85,93.8,2000,2019,40,9.0
9,sdg,Extended,Exclude flagged pairs,161,85,91.0,2000,2019,37,9.3


In [33]:
from pathlib import Path

PROJECT_ROOT = Path(
    r"D:\胡溪筼\Documents\2026-7Fall"
    r"\Carolina Data Challenge 2026"
)

PREPARED_DIR = PROJECT_ROOT / "Graduate_Dataset" / "prepared"
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

for source, frame in expanded_samples.items():
    output_path = PREPARED_DIR / f"time_tax_{source}_matched.csv.gz"

    frame.to_csv(
        output_path,
        index=False,
        compression="gzip",
    )

    print(f"Saved {len(frame)} rows: {output_path.name}")

source_conflicts.to_csv(
    PREPARED_DIR / "time_tax_source_conflicts.csv",
    index=False,
)

indicator_dictionary.to_csv(
    PREPARED_DIR / "time_tax_indicator_dictionary.csv",
    index=False,
)

Saved 66 rows: time_tax_gender_matched.csv.gz
Saved 177 rows: time_tax_sdg_matched.csv.gz
